# Phase 1: FashionCLIP Embedding & Style Anchor Generation

범용 CLIP ViT-B/32 → FashionCLIP (패션 80만장 fine-tuned)로 교체.
- FashionCLIP 로드
- 스타일 이미지로 앵커 벡터 생성 (5→10 스타일)
- 기존 CLIP vs FashionCLIP 비교 검증
- `anchors_v2.json` 생성 → 서버 배포

In [ ]:
!pip install -q transformers torch pillow numpy google-cloud-storage

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 1. FashionCLIP 모델 로드

In [ ]:
MODEL_NAME = "patrickjohncyh/fashion-clip"

processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
print(f"FashionCLIP loaded: {MODEL_NAME}")

## 2. 스타일 앵커 이미지 디렉토리 설정

Google Drive에 `data/style_anchor/` 폴더를 마운트하거나,
GCS에서 다운로드하세요.

In [ ]:
# Google Drive mount (Colab)
from google.colab import drive
drive.mount('/content/drive')

# 스타일 이미지 경로 설정
STYLE_DIR = Path("/content/drive/MyDrive/fashioncloset/data/style_anchor")
# 또는 로컬 업로드한 경우:
# STYLE_DIR = Path("/content/style_anchor")

# 10개 스타일
STYLES = [
    "casual", "formal", "minimal", "street", "vintage",
    "gorpcore", "workwear", "preppy", "romantic", "sporty",
]

print(f"Style dir: {STYLE_DIR}")
for s in STYLES:
    p = STYLE_DIR / s
    if p.exists():
        imgs = [f for f in p.iterdir() if f.suffix.lower() in {".jpg", ".jpeg", ".png"}]
        print(f"  {s}: {len(imgs)} images")
    else:
        print(f"  {s}: NOT FOUND (will use text anchor)")

## 3. 이미지 인코딩 함수

In [ ]:
def encode_image(image_path: str) -> np.ndarray:
    """이미지를 FashionCLIP 임베딩으로 인코딩."""
    img = Image.open(image_path).convert("RGB")
    inputs = processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        emb = model.get_image_features(**inputs)
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb[0].cpu().numpy().astype(np.float32)


def encode_text(text: str) -> np.ndarray:
    """텍스트를 FashionCLIP 임베딩으로 인코딩."""
    inputs = processor(text=[text], return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        emb = model.get_text_features(**inputs)
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb[0].cpu().numpy().astype(np.float32)

## 4. 스타일 앵커 생성

In [ ]:
# 텍스트 프롬프트 (이미지 없는 스타일용 fallback)
STYLE_TEXT_PROMPTS = {
    "casual": "casual everyday outfit, hoodie, jeans, sneakers, relaxed fit",
    "formal": "formal business outfit, suit, blazer, dress shoes, tailored",
    "minimal": "minimalist clean outfit, solid colors, simple silhouette, neutral tones",
    "street": "streetwear outfit, oversized, graphic tee, cargo pants, sneakers",
    "vintage": "vintage retro outfit, corduroy, tweed, washed denim, 90s style",
    "gorpcore": "gorpcore outdoor outfit, fleece jacket, hiking boots, cargo pants, technical wear",
    "workwear": "workwear utility outfit, denim jacket, canvas pants, work boots, heritage",
    "preppy": "preppy ivy league outfit, blazer, polo shirt, chinos, loafers",
    "romantic": "romantic feminine outfit, floral dress, lace blouse, pleated skirt, pastel",
    "sporty": "sporty athleisure outfit, track jacket, joggers, sneakers, mesh",
}

anchors = {}
TEXT_WEIGHT = 3.0  # 텍스트 프롬프트 가중치

for style in STYLES:
    img_dir = STYLE_DIR / style
    vectors = []
    
    # 이미지 인코딩
    if img_dir.exists():
        for img_path in sorted(img_dir.iterdir()):
            if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
                continue
            try:
                vec = encode_image(str(img_path))
                vectors.append(vec)
            except Exception as e:
                print(f"  SKIP {img_path.name}: {e}")
    
    # 텍스트 앵커 추가 (가중)
    text_prompt = STYLE_TEXT_PROMPTS.get(style, f"{style} fashion outfit")
    text_vec = encode_text(text_prompt)
    for _ in range(int(TEXT_WEIGHT)):
        vectors.append(text_vec)
    
    if vectors:
        centroid = np.mean(np.stack(vectors), axis=0)
        centroid = centroid / np.linalg.norm(centroid)
        anchors[style] = centroid
        print(f"  {style}: {len(vectors)} vectors (incl. text) → anchor dim={centroid.shape[0]}")
    else:
        print(f"  {style}: NO VECTORS")

print(f"\nTotal styles: {len(anchors)}")

## 5. anchors_v2.json 저장

In [ ]:
output = {
    "version": 2,
    "model": "patrickjohncyh/fashion-clip",
    "text_weight": TEXT_WEIGHT,
    "styles": list(anchors.keys()),
    "anchors": {k: v.tolist() for k, v in anchors.items()},
}

output_path = "/content/anchors_v2.json"
with open(output_path, "w") as f:
    json.dump(output, f)

file_size = os.path.getsize(output_path)
print(f"Saved: {output_path} ({file_size / 1024:.1f} KB)")

# Google Drive에도 저장
drive_path = "/content/drive/MyDrive/fashioncloset/data/style_anchor/anchors_v2.json"
with open(drive_path, "w") as f:
    json.dump(output, f)
print(f"Saved to Drive: {drive_path}")

## 6. 검증: 기존 CLIP vs FashionCLIP 분류 정확도 비교

In [ ]:
# 기존 CLIP 로드 (비교용)
!pip install -q git+https://github.com/openai/CLIP.git
import clip

clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()


def encode_image_clip(image_path: str) -> np.ndarray:
    img = clip_preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = clip_model.encode_image(img)
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb[0].cpu().numpy().astype(np.float32)

In [ ]:
# 테스트 이미지로 분류 정확도 비교
TEST_DIR = Path("/content/drive/MyDrive/fashioncloset/data/test_images")
# 각 스타일 폴더에 테스트 이미지가 있다고 가정

results = {"clip": {"correct": 0, "total": 0}, "fashionclip": {"correct": 0, "total": 0}}

# 기존 CLIP 앵커 로드
old_anchors_path = STYLE_DIR / "anchors.json"
if old_anchors_path.exists():
    with open(old_anchors_path) as f:
        old_data = json.load(f)
    old_anchors = {k: np.array(v, dtype=np.float32) for k, v in old_data["anchors"].items()}
else:
    old_anchors = {}
    print("No old anchors.json found")

for style in ["casual", "formal", "minimal", "street", "vintage"]:
    test_style_dir = TEST_DIR / style
    if not test_style_dir.exists():
        continue
    
    for img_path in sorted(test_style_dir.iterdir()):
        if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        
        # FashionCLIP
        fc_vec = encode_image(str(img_path))
        fc_sims = {s: float(np.dot(fc_vec, anchors[s])) for s in anchors if s in ["casual", "formal", "minimal", "street", "vintage"]}
        fc_pred = max(fc_sims, key=fc_sims.get)
        results["fashionclip"]["total"] += 1
        if fc_pred == style:
            results["fashionclip"]["correct"] += 1
        
        # Old CLIP
        if old_anchors:
            clip_vec = encode_image_clip(str(img_path))
            clip_sims = {s: float(np.dot(clip_vec, old_anchors[s])) for s in old_anchors}
            clip_pred = max(clip_sims, key=clip_sims.get)
            results["clip"]["total"] += 1
            if clip_pred == style:
                results["clip"]["correct"] += 1

print("=" * 50)
print("Style Classification Accuracy Comparison")
print("=" * 50)
for model_name, r in results.items():
    if r["total"] > 0:
        acc = r["correct"] / r["total"] * 100
        print(f"{model_name:15s}: {r['correct']}/{r['total']} = {acc:.1f}%")
    else:
        print(f"{model_name:15s}: no test images")

## 7. 서버 배포

`anchors_v2.json`을 서버의 `data/style_anchor/` 폴더에 복사하면 자동 적용됩니다.